# Baseline Model and Validation Strategy

In this notebook, we demonstrate the validation scheme and evaluate a naive baseline model.

The validation scheme scripts are located in `src/validation/`.

To evaluate the model's performance, we split the dataset into several test sets. In each test, the model is trained on all previous months and used to predict sales for the following month.

The baseline model is located in `src/models/`. It predicts the sales of an item in a shop using the previous month's sales. If the item was not sold in the previous month, the model predicts `0`.

---

## 1. Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import root_mean_squared_error

from src.data.etl import (
    ItemCategoryLoader,
    ItemsLoader,
    ShopsLoader,
    TrainLoader,
    DfFinalLoader,
    ForSubmissionLoader,
)

from src.models.baseline_model import BaseLineModel
from src.validation.chema import validation_chema


## 2. Data loading

In [2]:
project_root = Path.cwd().parent
raw_path = project_root / "data" / "raw"
processed_path = project_root / "data" / "processed"

items_df = ItemsLoader(raw_data_path=raw_path, cache_path=processed_path).load()
item_cats_df = ItemCategoryLoader(raw_data_path=raw_path, cache_path=processed_path).load()
shops_df = ShopsLoader(raw_data_path=raw_path, cache_path=processed_path).load()
train_df = TrainLoader(raw_data_path=raw_path, cache_path=processed_path).load()
final_df = DfFinalLoader(cache_path=processed_path, shops=shops_df, items=items_df, item_cats=item_cats_df, train=train_df).load()
subm_df = ForSubmissionLoader(raw_data_path=raw_path, cache_path=processed_path, shops=shops_df, items=items_df).load()

## 4. Creating model

Lets compare results of this model to constants

We will use RMSE as this is a metric used in Kaggle competition

In [3]:
results = validation_chema(final_df, BaseLineModel, root_mean_squared_error)

In [4]:
class ZerosModel:
    def __init__(self):
        pass

    def fit(self, df):
        return self

    def predict(self, df):
        return [0] * len(df)

results_zeros = validation_chema(final_df, ZerosModel, root_mean_squared_error)

In [5]:
class OnesModel:
    def __init__(self):
        pass

    def fit(self, df):
        return self

    def predict(self, df):
        return [1] * len(df)

results_ones = validation_chema(final_df, OnesModel, root_mean_squared_error)

In [6]:
compare_df = pd.DataFrame({'zeros': results_zeros, 'ones': results_ones, 'baseline_model': results})
compare_df.index = pd.date_range('2013-02-01', periods=len(compare_df), freq='MS').strftime('%Y-%m')
compare_df

,zeros,ones,baseline_model
2013-02,4.329915,3.933574,3.952906
2013-03,5.358890,5.011686,5.041874
2013-04,3.854385,3.454272,4.806216
2013-05,5.716203,5.446137,5.434950
2013-06,9.155228,8.964116,6.796802
2013-07,7.437187,7.230517,4.170925
2013-08,7.922950,7.710654,4.207923
2013-09,11.385041,11.200379,8.335206
2013-10,9.883269,9.679148,7.410759
2013-11,10.265728,10.066368,6.198169


In [7]:
compare_df.mean()

zeros             8.519044
ones              8.289650
baseline_model    6.609165
dtype: float64

We can see that overall this model predicts better then constants but after spikes like new year this model has high expectations and looses to constants that keep sales low.